<a href="https://colab.research.google.com/github/JoshuaNiel/CS452/blob/main/sparksql/03%20-%20Covid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup Java
!apt-get install openjdk-11-jdk -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-

In [2]:
# Setup Spark SQL
# Note if running locally you need the JVM https://www.oracle.com/java/technologies/downloads/
# Also, if running locally you'll need to allow it to talk over the network to your own machine
# Consider running in https://colab.research.google.com/
%pip install pyspark

In [3]:
# Initialize Context - this is where you'd setup information about your Hadoop cluster if you had one!
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("Covid").getOrCreate()

sc = spark.sparkContext

sc.setLogLevel("WARN")

In [4]:
# Download 100mb covid county data file
!curl "https://raw.githubusercontent.com/nytimes/covid-19-data/master/us-counties.csv" > ./uscounties.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 99.9M  100 99.9M    0     0  43.7M      0  0:00:02  0:00:02 --:--:-- 43.7M


In [23]:
# Write code to define or infer the schema and then read in the dataset
# Question 1
# Read the file into a Spark DataFrame
usCountiesFilePath = "./uscounties.csv"
from pyspark.sql.types import DateType, StringType, StructField, StructType, IntegerType

# the fips column needs to be a string in order to preserve leading zeros
schema = StructType([StructField('date', DateType(), True), StructField('county', StringType(), True), StructField('state', StringType(), True), StructField('fips', StringType(), True), StructField('cases', IntegerType(), True), StructField('deaths', IntegerType(), True)])

df = spark.read.csv(usCountiesFilePath, schema=schema, header=True)

In [24]:
# Write code to find the county with the most deaths
# Option 1) SparkSQL API
df.createOrReplaceTempView("covid")  # create table that you can do sql on

# Question 2
print("Max deaths:")
spark.sql(
    """
    select county, state, deaths
    from covid
    order by deaths desc
    limit 1
  """
).show()

Max deaths:
+-------------+--------+------+
|       county|   state|deaths|
+-------------+--------+------+
|New York City|New York| 40267|
+-------------+--------+------+



In [9]:
# Option 2) # DataFrame style
from pyspark.sql.functions import col

print("Max deaths:")
print(
    df.orderBy(col("deaths").desc()).take(  # .where(col("county") == "New York City") \
        1
    )
)

Max deaths:
[Row(date=datetime.date(2022, 5, 13), county='New York City', state='New York', fips=None, cases=2422658, deaths=40267)]


In [10]:
# # Option 3) RDD MapReduce Style without key
rows = df.rdd


def getMax(cumm, other):
    if other["deaths"] is not None and other["deaths"] > cumm["deaths"]:
        return other
    else:
        return cumm


print("Max deaths:")
print(rows.reduce(getMax))

Max deaths:
Row(date=datetime.date(2022, 5, 13), county='New York City', state='New York', fips=None, cases=2422658, deaths=40267)


In [11]:
# # Option 4) RDD MapReduce Style with mapped tuples
rows = df.rdd


def getMax(cumm, other):
    if other[0] > cumm[0]:
        return other
    else:
        return cumm


rows = rows.map(lambda r: (r["deaths"] or 0, f"{r['county']},{r['state']}"))
print("Max deaths:")
print(rows.reduce(getMax))

Max deaths:
(40267, 'New York City,New York')


In [12]:
# Write code to find the county with the most deaths
# Question 2
print("Max deaths:")
spark.sql(
    """
    select county, state, deaths
    from covid
    order by deaths desc
    limit 1
  """
).show()

Max deaths:
+-------------+--------+------+
|       county|   state|deaths|
+-------------+--------+------+
|New York City|New York| 40267|
+-------------+--------+------+



In [36]:
# Write code to find the county with the most cases
# Question 3
print("Max cases:")
spark.sql(
    """
    select county, state, cases
    from covid
    order by cases desc
    limit 1
  """
).show()

Max cases:
+-----------+----------+-------+
|     county|     state|  cases|
+-----------+----------+-------+
|Los Angeles|California|2908425|
+-----------+----------+-------+



In [37]:
# Write code to find the total number of deaths in Utah county
# Question 4
print("Total deaths in Utah county:")
spark.sql("""
  SELECT county, state, deaths
  FROM covid
  WHERE county = 'Utah'
  ORDER BY date DESC
  LIMIT 1
""").show()

Total deaths in Utah county:
+------+-----+------+
|county|state|deaths|
+------+-----+------+
|  Utah| Utah|   791|
+------+-----+------+



In [67]:
# Write code to find the death rate for each state and sort the states by death rate descending
# Question 5
print("Death rate per state:")
death_rate_df = spark.sql("""
  WITH recent_dates AS (
    SELECT county, state, MAX(date) AS recent_date
    FROM covid
    GROUP BY county, state
  )
  SELECT c.state, SUM(c.deaths) AS total_state_deaths, SUM(c.cases)  AS total_state_cases, (total_state_deaths / total_state_cases * 100) AS death_rate
  FROM covid c
  INNER JOIN recent_dates rd ON c.county = rd.county AND c.state = rd.state AND c.date = rd.recent_date
  GROUP BY c.state
  ORDER BY death_rate DESC
""")
death_rate_df.show()

Death rate per state:
+-------------+------------------+-----------------+------------------+
|        state|total_state_deaths|total_state_cases|        death_rate|
+-------------+------------------+-----------------+------------------+
| Pennsylvania|             44814|          2850361|1.5722219045236727|
|  Mississippi|             12457|           801527|1.5541584999631952|
|      Alabama|             19628|          1304710|1.5043956128181741|
|       Nevada|             10802|           724922|1.4900913477587934|
|      Arizona|             30230|          2030942|1.4884718519780475|
|      Georgia|             36605|          2460845|1.4874971808464166|
|     Michigan|             36140|          2472596|1.4616217125644464|
|   New Jersey|             33537|          2313062| 1.449896284665089|
|   New Mexico|              7609|           526137|1.4462012745729724|
|         Ohio|             38550|          2724041|1.4151769374983711|
|     Missouri|             20586|        

In [66]:
# Write code to something else interesting with this data – your choice
print("Smallest difference between cases and deaths on a certain day")
spark.sql("""
  SELECT COUNT(*) AS days_with_perfect_kill_rate
  FROM covid
  WHERE deaths IS NOT NULL AND cases IS NOT NULL and county != 'Unknown' AND cases != 0 AND deaths != 0 AND deaths >= cases
""").show()

Smallest difference between cases and deaths on a certain day
+---------------------------+
|days_with_perfect_kill_rate|
+---------------------------+
|                        450|
+---------------------------+



In [71]:
# Extra Credit 1 - Plot your death rate data!
# Extra Credit 2 - Join this with other data or find something intresting in this data and plot it on a map!
import pandas as pd
import plotly.express as px

us_state_to_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC", "American Samoa": "AS", "Guam": "GU", "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR", "United States Virgin Islands": "VI",
}
death_rate_df = death_rate_df.toPandas()

death_rate_df['state_code'] = death_rate_df['state'].map(us_state_to_abbrev)


fig = px.choropleth(
    death_rate_df,
    locations='state_code', # Column with state abbreviations
    locationmode='USA-states', # Set location mode to US states
    color='death_rate', # Column to determine color intensity
    scope='usa', # Limit map to the USA
    color_continuous_scale='Viridis', # Choose a color scale
    title='Extra Credit Plot Death Rate'
)

fig.show()

In [ ]:
# This example uses two-letter state code

import pandas as pd
import plotly.express as px

data = pd.DataFrame({
  'state': ['NY', 'CA', 'TX', 'FL'],
  'values': [10, 20, 15, 25]
})

fig = px.choropleth(
    data,
    locations='state', # Column with state abbreviations
    locationmode='USA-states', # Set location mode to US states
    color='values', # Column to determine color intensity
    scope='usa', # Limit map to the USA
    color_continuous_scale='Viridis', # Choose a color scale
    title='Extra Credit Plot <Insert name here>'
)

fig.show()


In [ ]:
# This example uses the FIPS code to map data to a county

import pandas as pd
import plotly.express as px

# Example county-level data (FIPS codes are required for county-level plots)
data = pd.DataFrame({
    'fips': ['36061', '06037', '48201', '12086'],  # Example FIPS codes (NYC, LA, Houston, Miami-Dade)
    'values': [10, 20, 15, 25]
})

# Plot county-level choropleth map
fig = px.choropleth(
    data,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",  # GeoJSON for counties
    locations='fips',  # Use county FIPS codes
    color='values',  # Column to determine color intensity
    color_continuous_scale='Viridis',
    scope='usa',
    title='County-Level Extra Credit Plot'
)

fig.show()
